# JEPA-for-Trading V5

Action-Primitive JEPA world model planner. The market transition is action-independent, while portfolio consequences are action-conditioned through primitive trading intentions projected by one execution layer. The planner uses MPC/CEM over primitive actions, executes one constrained action, then replans the next day.


In [ ]:
# Kaggle/bootstrap cell. Run from /kaggle/working.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/aurvl/jepa-for-trading.git"
BRANCH = "version5"
REPO_DIR = Path("/kaggle/working/jepa-for-trading")

if not REPO_DIR.exists():
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!{sys.executable} -m pip install -q -e . --no-deps
!git log --oneline -3


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from jepa_trading.config import ensure_dirs, load_config
from jepa_trading.data.pipeline import prepare_market_data, create_v5_dataloaders
from jepa_trading.data.v2_dataset import PortfolioActionConfig
from jepa_trading.actions import primitive_action_dim
from jepa_trading.data.v5_dataset import V5CostConfig, summarize_v5_batch
from jepa_trading.evaluation.backtest import (
    buy_and_hold_weight, equal_weight, momentum_weight, random_long_only_weight,
    run_weight_strategy, volatility_target_weight,
)
from jepa_trading.evaluation.metrics import metrics_table, validate_backtest_histories
from jepa_trading.evaluation.v5_backtest import run_v5_planner_backtest, validate_v5_backtest
from jepa_trading.models.world_model_v5 import V5ActionConditionedWorldModel
from jepa_trading.planning.v5_planner import V5ActionWorldModelPlanner
from jepa_trading.rl.env import TradingEnv
from jepa_trading.rl.observer import RawMarketObserver
from jepa_trading.training.checkpoints import load_checkpoint
from jepa_trading.training.train_v5 import train_v5_world_model
from jepa_trading.utils.device import get_device
from jepa_trading.utils.seed import seed_everything


In [ ]:
config = load_config('configs/default.yaml')

# Kaggle macro auto-discovery: supports macro_data.parquet or estimated_volatility_with_macro.csv.
macro_candidates = []
if Path('/kaggle/input').exists():
    macro_candidates = list(Path('/kaggle/input').rglob('macro_data.parquet'))
    macro_candidates += list(Path('/kaggle/input').rglob('estimated_volatility_with_macro.csv'))
if macro_candidates:
    config['data']['macro_path'] = str(macro_candidates[0])

FAST_DEV_RUN = False
if FAST_DEV_RUN:
    config['data']['tickers'] = config['data']['tickers'][:8]
    config['v5']['world_max_steps'] = 20
    config['training']['eval_every'] = 10
    config['v5']['planner_sampled_actions'] = 16
    config['ppo']['total_updates'] = 2

ensure_dirs(config)
seed_everything(config['seed'])
device = get_device(config['device'])
print(device)
print('macro_path:', config['data']['macro_path'])


In [ ]:
prepared_df, arrays, feature_columns = prepare_market_data(config, force_download=False)
loaders = create_v5_dataloaders(config, arrays)
batch = next(iter(loaders['train']))
print('rows:', len(prepared_df))
print('assets:', len(arrays.tickers), arrays.tickers)
print('features:', len(feature_columns), feature_columns)
print('dataset sizes:', {k: len(v.dataset) for k, v in loaders.items()})
print('batch summary:', summarize_v5_batch(batch))


In [ ]:
portfolio_cfg = config['portfolio']
v5_cfg = config['v5']

model = V5ActionConditionedWorldModel(
    n_features=len(feature_columns),
    max_assets=len(arrays.tickers),
    portfolio_state_dim=batch['portfolio_state'].shape[-1],
    primitive_action_dim=batch['primitive_actions'].shape[-1],
    executable_action_dim=batch['executable_actions'].shape[-1],
    goal_dim=batch['goal'].shape[-1],
    d_model=config['model']['d_model'],
    latent_dim=config['model']['latent_dim'],
    n_heads=config['model']['n_heads'],
    n_layers=config['model']['n_layers'],
    dropout=config['model']['dropout'],
    ema_decay=config['training']['ema_decay'],
    hidden_dim=config['v2'].get('hidden_dim', 256),
    asset_embedding_dim=config['model'].get('asset_embedding_dim'),
)

OUTPUT_DIR = Path('outputs/v5_latest')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
v5_ckpt = Path(config['training']['checkpoint_dir']) / 'v5_action_world_model.pt'
v5_history_path = OUTPUT_DIR / 'v5_training_history.csv'

if v5_ckpt.exists():
    print('Loading existing V5 checkpoint:', v5_ckpt)
    metadata = load_checkpoint(v5_ckpt, model, map_location=device)
    print('checkpoint metadata:', metadata)
    v5_history = pd.DataFrame([{"event": "checkpoint_loaded_no_new_training", **metadata}])
    v5_history.to_csv(v5_history_path, index=False)
else:
    v5_history = train_v5_world_model(
        model=model,
        train_loader=loaders['train'],
        val_loader=loaders['val'],
        device=device,
        max_steps=v5_cfg['world_max_steps'],
        warmup_steps=v5_cfg['world_warmup_steps'],
        lr=config['training']['lr'],
        weight_decay=config['training']['weight_decay'],
        checkpoint_path=v5_ckpt,
        weights=v5_cfg['loss_weights'],
        rank_margin=v5_cfg['rank_margin'],
        eval_every=config['training']['eval_every'],
        log_every=config['training']['log_every'],
        history_path=v5_history_path,
        history_save_every=config['training'].get('log_every', 25),
    )
    display(v5_history.tail())
    load_checkpoint(v5_ckpt, model, map_location=device)

model.to(device).eval()
print('history saved to:', v5_history_path)
print('model ready')


In [ ]:
action_cfg = PortfolioActionConfig(
    mode=portfolio_cfg.get('mode', 'long_only'),
    cash_initial=portfolio_cfg['cash_initial'],
    transaction_cost_bps=portfolio_cfg['transaction_cost_bps'],
    max_long_weight=portfolio_cfg.get('max_long_weight', portfolio_cfg.get('max_weight_per_asset', 0.15)),
    max_short_weight=portfolio_cfg.get('max_short_weight', 0.05),
    max_gross_exposure=portfolio_cfg.get('max_gross_exposure', 1.0),
    max_net_exposure=portfolio_cfg.get('max_net_exposure', 1.0),
    borrow_cost_bps=portfolio_cfg.get('borrow_cost_bps', 2.0),
    n_action_samples=v5_cfg['planner_sampled_actions'],
    derisk_fraction=portfolio_cfg.get('derisk_fraction', 0.50),
)
cost_cfg = V5CostConfig(**v5_cfg['cost'])
planner = V5ActionWorldModelPlanner(
    model=model,
    action_config=action_cfg,
    cost_config=cost_cfg,
    horizons=config['data']['horizons'],
    n_sampled_actions=v5_cfg['planner_sampled_actions'],
    max_turnover=portfolio_cfg.get('max_turnover', 0.40),
    cem_iters=v5_cfg.get('cem_iters', 3),
    elite_frac=v5_cfg.get('cem_elite_frac', 0.20),
    uncertainty_penalty=v5_cfg.get('uncertainty_penalty', 0.15),
    turnover_penalty=v5_cfg.get('turnover_penalty', 0.02),
    device=device,
    seed=config['seed'],
)
print('Loaded', planner.CODE_VERSION)


In [ ]:
test_start = pd.Timestamp(config['data']['val_end']) + pd.Timedelta(days=1)
test_end = arrays.dates.max()
agent_history = run_v5_planner_backtest(
    arrays=arrays,
    planner=planner,
    start_date=test_start,
    end_date=test_end,
    lookback=config['data']['lookback'],
    cash_initial=portfolio_cfg['cash_initial'],
    transaction_cost_bps=portfolio_cfg['transaction_cost_bps'],
    max_weight_per_asset=portfolio_cfg['max_weight_per_asset'],
    max_turnover=portfolio_cfg['max_turnover'],
    mode=portfolio_cfg.get('mode', 'long_only'),
    max_long_weight=portfolio_cfg.get('max_long_weight', portfolio_cfg.get('max_weight_per_asset', 0.15)),
    max_short_weight=portfolio_cfg.get('max_short_weight', 0.05),
    max_gross_exposure=portfolio_cfg.get('max_gross_exposure', 1.0),
    max_net_exposure=portfolio_cfg.get('max_net_exposure', 1.0),
    borrow_cost_bps=portfolio_cfg.get('borrow_cost_bps', 2.0),
    output_dir=OUTPUT_DIR,
)
agent_history.tail()


In [ ]:
observer = RawMarketObserver(arrays, config['data']['lookback'])
def make_test_env():
    return TradingEnv(
        arrays, observer, start_date=test_start, end_date=test_end, lookback=config['data']['lookback'],
        cash_initial=portfolio_cfg['cash_initial'],
        transaction_cost_bps=portfolio_cfg['transaction_cost_bps'],
        max_weight_per_asset=portfolio_cfg['max_weight_per_asset'],
        max_turnover=portfolio_cfg['max_turnover'],
        mode=portfolio_cfg.get('mode', 'long_only'),
        max_long_weight=portfolio_cfg.get('max_long_weight', portfolio_cfg.get('max_weight_per_asset', 0.15)),
        max_short_weight=portfolio_cfg.get('max_short_weight', 0.05),
        max_gross_exposure=portfolio_cfg.get('max_gross_exposure', 1.0),
        max_net_exposure=portfolio_cfg.get('max_net_exposure', 1.0),
        borrow_cost_bps=portfolio_cfg.get('borrow_cost_bps', 2.0),
    )

buy_hold = run_weight_strategy(make_test_env(), buy_and_hold_weight)
equal_hist = run_weight_strategy(make_test_env(), equal_weight)
momentum_hist = run_weight_strategy(make_test_env(), momentum_weight)
vol_target_hist = run_weight_strategy(make_test_env(), volatility_target_weight)
rng = np.random.default_rng(config['seed'])
random_histories = [run_weight_strategy(make_test_env(), lambda env, mask, rng=rng: random_long_only_weight(env, mask, rng)) for _ in range(100)]

histories = {
    'V5 Action JEPA World Model': agent_history,
    'Buy & Hold': buy_hold,
    'Equal Weight': equal_hist,
    'Momentum': momentum_hist,
    'Vol Target': vol_target_hist,
}
validity = validate_backtest_histories(histories)
metrics = metrics_table(histories)
print(validity.to_string(index=False))
display(metrics)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
agent_history.to_csv(OUTPUT_DIR / 'agent_history.csv', index=False)
buy_hold.to_csv(OUTPUT_DIR / 'buy_hold_history.csv', index=False)
equal_hist.to_csv(OUTPUT_DIR / 'equal_weight_history.csv', index=False)
momentum_hist.to_csv(OUTPUT_DIR / 'momentum_history.csv', index=False)
vol_target_hist.to_csv(OUTPUT_DIR / 'vol_target_history.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
validity.to_csv(OUTPUT_DIR / 'baseline_validity.csv', index=False)
v5_validity = validate_v5_backtest(agent_history, 'V5 Action-Primitive JEPA World Model')
v5_validity.to_csv(OUTPUT_DIR / 'validity.csv', index=False)
agent_history['selected_action_name'].value_counts().rename_axis('action').reset_index(name='count').to_csv(OUTPUT_DIR / 'action_distribution.csv', index=False)
if 'planned_horizon' in agent_history:
    agent_history['planned_horizon'].value_counts().rename_axis('horizon').reset_index(name='count').to_csv(OUTPUT_DIR / 'horizon_distribution.csv', index=False)
primitive_cols = [c for c in agent_history.columns if c.startswith('primitive_')]
if primitive_cols:
    agent_history[primitive_cols].describe().T.to_csv(OUTPUT_DIR / 'primitive_action_stats.csv')
pred_cols = [c for c in agent_history.columns if c.startswith('pred_')]
if pred_cols:
    agent_history[['date', *pred_cols]].to_csv(OUTPUT_DIR / 'calibration_report.csv', index=False)
if 'prediction_error' in agent_history:
    agent_history[['date', 'prediction_error']].to_csv(OUTPUT_DIR / 'prediction_error.csv', index=False)

random_rows = []
for i, hist in enumerate(random_histories):
    hist.to_csv(OUTPUT_DIR / f'random_history_{i:03d}.csv', index=False)
    random_rows.append({
        'random_id': i,
        'final_equity': float(hist['equity'].iloc[-1]),
        'total_return': float(hist['equity'].iloc[-1] / hist['equity'].iloc[0] - 1.0),
    })
pd.DataFrame(random_rows).to_csv(OUTPUT_DIR / 'random_summary.csv', index=False)

summary = {
    'branch': 'version5',
    'planner_code_version': planner.CODE_VERSION,
    'checkpoint': str(v5_ckpt),
    'n_assets': len(arrays.tickers),
    'n_features': len(feature_columns),
    'test_start': str(test_start),
    'test_end': str(test_end),
    'action_distribution': agent_history['selected_action_name'].value_counts().to_dict(),
    'avg_cash_weight': float(agent_history['cash_weight'].mean()),
    'avg_gross_exposure': float(agent_history['gross_exposure'].mean()),
    'avg_turnover': float(agent_history['turnover'].mean()),
    'final_equity': float(agent_history['equity'].iloc[-1]),
    'training_history_file': str(v5_history_path),
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(16, 7))
for hist in random_histories:
    ax.plot(hist['date'], hist['equity'], color='gray', alpha=0.16, linewidth=0.8)
ax.plot(equal_hist['date'], equal_hist['equity'], label='Equal Weight')
ax.plot(momentum_hist['date'], momentum_hist['equity'], label='Momentum')
ax.plot(vol_target_hist['date'], vol_target_hist['equity'], label='Vol Target')
ax.plot(buy_hold['date'], buy_hold['equity'], color='white', linewidth=2.0, label='Buy & Hold')
ax.plot(agent_history['date'], agent_history['equity'], color='#39ff14', linewidth=2.5, label='V5 Action JEPA WM')
ax.set_title('Trading Strategy Equity Curves')
ax.set_xlabel('Date')
ax.set_ylabel('Equity')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'equity_curves.png', dpi=160)
plt.show()

print('V5 validity:')
print(v5_validity.to_string(index=False))
print('Saved outputs to', OUTPUT_DIR)


In [ ]:
# Optional Kaggle push cell. Requires a valid GITHUB_TOKEN secret.
import os, subprocess
from pathlib import Path

BRANCH = 'version5'
TOKEN = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not TOKEN:
    print('No token found. Add GITHUB_TOKEN in Kaggle Secrets to push outputs.')
else:
    def run(cmd, check=True):
        print('$', ' '.join(cmd))
        return subprocess.run(cmd, check=check, text=True)
    run(['git', 'config', 'user.email', 'aurelvhei@outlook.fr'])
    run(['git', 'config', 'user.name', 'aurvl'])
    run(['git', 'remote', 'set-url', 'origin', 'https://' + 'x-' + f'access-token:{TOKEN}@github.com/aurvl/jepa-for-trading.git'])
    run(['git', 'add', '.'])
    # outputs/*, models/* and logs/* are ignored by .gitignore to avoid accidental huge commits.
    # Force-add the V5 run artifacts explicitly so Kaggle results are actually pushed.
    for path in ['outputs/v5_latest', 'models/v5_action_world_model.pt', 'logs/v5_training_history.csv']:
        if Path(path).exists():
            run(['git', 'add', '-f', path])
    run(['git', 'status', '--short'])
    run(['git', 'commit', '-m', 'Add V5 Kaggle run artifacts'], check=False)
    run(['git', 'push', 'origin', f'HEAD:{BRANCH}'])
    run(['git', 'remote', 'set-url', 'origin', 'https://github.com/aurvl/jepa-for-trading.git'], check=False)
